In [2]:
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import numpy as np
import random
from time import time
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn import model_selection, metrics
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
def display_cm(confusion_matrix):
    cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix=confusion_matrix)
    cm_display.plot()
    plt.show()


def calculate_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    accuracy = np.sum(pred_flat == labels_flat) / len(labels_flat)
    return accuracy


def train_transformer_with_weights(input: pd.Series,
                      label: pd.Series,
                      model_name: str,
                      saved_model_name: str,
                      class_weights_tensor,
                      num_labels=10,
                      max_length=512,
                      val_size=0.1,
                      batch_size=16,
                      epochs=2,
                      learning_rate=2e-5):
    # Set seed
    seed = 100
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    model.to(device)

    # Encode
    encodings = tokenizer.batch_encode_plus(
        input.to_list(),
        max_length=max_length, # NOTE: choose "max_length" according to input sequence length distribution - see if it affects results
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    input_ids = encodings["input_ids"]
    attention_masks = encodings["attention_mask"]
    labels = torch.tensor(label.values)

    # Train and validation Dataloaders

    train_ids, val_ids, train_masks, val_masks, train_labels, val_labels = train_test_split(
        input_ids,
        attention_masks,
        labels,
        test_size=val_size,
        random_state=seed) # NOTE: Check whether shuffle and stratify changes results?

    train_data = TensorDataset(train_ids, train_masks, train_labels)
    val_data = TensorDataset(val_ids, val_masks, val_labels)
    g = torch.Generator()
    g.manual_seed(seed)
    train_dataloader = DataLoader(train_data, shuffle=True, batch_size=batch_size, generator=g) # NOTE: Check whether shuffle and batch changes results?
    val_dataloader = DataLoader(val_data, batch_size=batch_size)

    # Optimizer, loss function, scheduler
    # Schedulers are designed to gradually decrease the learning rate as the training continues
    optimizer = AdamW(model.parameters(), lr=learning_rate)
    training_steps = epochs * len(train_dataloader)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * training_steps), num_training_steps=training_steps)

    # Training

    graph_train_loss = []
    graph_train_accuracy = []
    graph_val_loss = []
    graph_val_accuracy = []
    graph_val_f1 = []
    class_weight_tensor = class_weights_tensor.to(device)
    loss_function = torch.nn.CrossEntropyLoss(weight=class_weight_tensor)
    for epoch in range(epochs):
        t1 = time()
        model.train() # train mode
        training_loss = 0
        training_accuracy = 0
        
        for batch in train_dataloader:
            batch_input_ids = batch[0].to(device)
            batch_attention_mask = batch[1].to(device)
            batch_labels = batch[2].to(device)

            model.zero_grad() # reset the calculated gradients from the previous iteration of this loop

            outputs = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask)
            logits = outputs.logits
            
            loss = loss_function(logits, batch_labels)
        
            training_loss += loss.item() # extract the float value using the item method
            loss.backward() # Perform a backward pass of the model and propagate the loss through the classifier head
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Clip the gradients to be no larger than 1.0 so the model does not suffer from the exploding gradients problem.
            optimizer.step() # call optimizer to make a steo in the direction of the error
            scheduler.step()
            training_accuracy += calculate_accuracy(logits.detach().cpu().numpy(), batch_labels.cpu().numpy())

        average_train_loss = training_loss / len(train_dataloader)
        average_train_accuracy = training_accuracy / len(train_dataloader)
        t2 = time()
        print(f"Epoch {epoch+1}/{epochs}")
        print(f"Average training loss: {average_train_loss:.4f}, average training accuracy: {average_train_accuracy:.4f}")
        print("Training time:", (t2-t1)/60)
        graph_train_loss.append(average_train_loss)
        graph_train_accuracy.append(average_train_accuracy)

        # Save model - doing checkpointing
        output_dir = f"./saved_model_{saved_model_name}_epoch{epoch+1}"
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)

        # Validation
        model.eval()
        val_loss = 0
        val_accuracy = 0
        true_labels = []
        predictions = []

        with torch.no_grad():
            for batch in val_dataloader:
                batch_input_ids = batch[0].to(device)
                batch_attention_mask = batch[1].to(device)
                batch_labels = batch[2].to(device)
                outputs = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask, labels=batch_labels)
                loss = outputs.loss
                logits = outputs.logits
                val_loss += loss.item()
                val_accuracy += calculate_accuracy(logits.detach().cpu().numpy(), batch_labels.cpu().numpy())

                preds = torch.argmax(logits, dim=1)
                true_labels.extend(batch_labels.cpu().numpy())
                predictions.extend(preds.cpu().numpy())

        average_val_accuracy = val_accuracy / len(val_dataloader)
        average_val_loss = val_loss / len(val_dataloader)
        f1score_val = metrics.f1_score(true_labels, predictions, average="macro")
        t3 = time()

        print(f"Avg val loss: {average_val_loss:.4f}, avg val accuracy: {average_val_accuracy:.4f}, f1 val (macro): {f1score_val}")
        print("Total epoch time", (t3-t1)/60)

        graph_val_loss.append(average_val_loss)
        graph_val_accuracy.append(average_val_accuracy)
        graph_val_f1.append(f1score_val)
        
    df = pd.DataFrame({"epochs": pd.Series(range(1, epochs+1)),
                       'train_loss': pd.Series(graph_train_loss), 
                       'val_loss': pd.Series(graph_val_loss),
                      "train_accuracy": pd.Series(graph_train_accuracy),
                      "val_accuracy": pd.Series(graph_val_accuracy),
                      "val_f1": pd.Series(graph_val_f1)})
    df.to_csv(f"./{saved_model_name}_train_val_metrics.csv", index=False)


    plt.plot(range(1, epochs+1), graph_train_accuracy, label="Training accuracy")
    plt.plot(range(1, epochs+1), graph_val_accuracy, label="Validation accuracy")
    plt.title("Training and validation accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend(loc='best')
    plt.show()

    plt.plot(range(1, epochs+1), graph_val_f1, label="Validation F1 score")
    plt.title("Validation F1 score")
    plt.xlabel("Epochs")
    plt.ylabel("F1")
    plt.legend(loc='best')
    plt.show()

    return model, tokenizer


# Predict on test set
def evaluate_model(inputs, label, model, tokenizer, max_length=512, batch_size=16):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    encodings = tokenizer.batch_encode_plus(
        inputs.to_list(),
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    input_ids = encodings["input_ids"]
    attention_masks = encodings["attention_mask"]
    labels = torch.tensor(label.values)
    test_data = TensorDataset(input_ids, attention_masks, labels)
    test_dataloader = DataLoader(test_data, batch_size=batch_size)
    all_preds = []
    all_labels = []

    with torch.no_grad():
            for batch in test_dataloader:
                batch_input_ids = batch[0].to(device)
                batch_attention_mask = batch[1].to(device)
                batch_labels = batch[2].to(device)
                outputs = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask, labels=batch_labels)
                logits = outputs.logits
                preds = torch.argmax(logits, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(batch_labels.cpu().numpy())

    print("Test Accuracy:", accuracy_score(all_labels, all_preds))
    print("Classification report:")
    print(classification_report(all_labels, all_preds, digits=4))
    conf_mat = confusion_matrix(all_labels, all_preds)
    display_cm(conf_mat)


### Fine-tuning

##### LitLat

In [ ]:
df = pd.read_csv("single_label.csv")
x_train, x_test, y_train, y_test = model_selection.train_test_split(df["fragment"], df["label"],
                                                                        random_state=100, test_size=0.2,
                                                                        stratify=df["label"])


class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

mod, tok = train_transformer_with_weights(x_train, y_train, 
                             model_name="EMBEDDIA/litlat-bert", 
                             saved_model_name="continued_litlat_weights",
                             class_weights_tensor=class_weights_tensor, 
                             epochs=20)


##### XLMR

In [ ]:
df = pd.read_csv("single_label.csv")
x_train, x_test, y_train, y_test = model_selection.train_test_split(df["fragment"], df["label"],
                                                                        random_state=100, test_size=0.2,
                                                                        stratify=df["label"])


class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

mod, tok = train_transformer_with_weights(x_train, y_train, 
                             model_name="xlm-roberta-base", 
                             saved_model_name="continued_xlmr_weights",
                             class_weights_tensor=class_weights_tensor, 
                             epochs=20)


##### mBERT

In [ ]:
df = pd.read_csv("single_label.csv")
x_train, x_test, y_train, y_test = model_selection.train_test_split(df["fragment"], df["label"],
                                                                        random_state=100, test_size=0.2,
                                                                        stratify=df["label"])


class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

mod, tok = train_transformer_with_weights(x_train, y_train, 
                             model_name="bert-base-multilingual-cased", 
                             saved_model_name="continued_mbert_weights",
                             class_weights_tensor=class_weights_tensor, 
                             epochs=20)

### Evaluate weighted models

In [ ]:
df = pd.read_csv("single_label.csv")
x_train, x_test, y_train, y_test = model_selection.train_test_split(df["fragment"], df["label"],
                                                                        random_state=100, test_size=0.2,
                                                                        stratify=df["label"])

model_ckp = "saved_model_continued_mbert_weights_epoch6"
mod = AutoModelForSequenceClassification.from_pretrained(model_ckp)
tok = AutoTokenizer.from_pretrained(model_ckp)
print("mBERT on 6th")
evaluate_model(x_test, y_test, mod, tok)

model_ckp = "saved_model_continued_xlmr_weights_epoch6"
mod = AutoModelForSequenceClassification.from_pretrained(model_ckp)
tok = AutoTokenizer.from_pretrained(model_ckp)
print("XLM-R")
evaluate_model(x_test, y_test, mod, tok)

model_ckp = "saved_model_continued_litlat_weights_epoch6"
mod = AutoModelForSequenceClassification.from_pretrained(model_ckp)
tok = AutoTokenizer.from_pretrained(model_ckp)
print("LitLat")
evaluate_model(x_test, y_test, mod, tok)

### Save best model predictions for XAI

In [ ]:
def extract_predictions(inputs, label, model, tokenizer, max_length=512, batch_size=16):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    encodings = tokenizer.batch_encode_plus(
        inputs.to_list(),
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    input_ids = encodings["input_ids"]
    attention_masks = encodings["attention_mask"]
    labels = torch.tensor(label.values)
    test_data = TensorDataset(input_ids, attention_masks, labels)
    test_dataloader = DataLoader(test_data, batch_size=batch_size)
    all_preds = []
    all_labels = []
    all_texts = []

    with torch.no_grad():
            for i, batch in enumerate(test_dataloader):
                batch_input_ids = batch[0].to(device)
                batch_attention_mask = batch[1].to(device)
                batch_labels = batch[2].to(device)
                outputs = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask, labels=batch_labels)
                logits = outputs.logits
                preds = torch.argmax(logits, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(batch_labels.cpu().numpy())

                start_index = i * batch_size
                end_index = start_index + len(batch_input_ids)
                text = inputs[start_index:end_index].to_list()
                all_texts.extend(text)

    df = pd.DataFrame({"Fragments": all_texts, "True_labels": all_labels, "Predicted_labels": all_preds})

    return df



In [ ]:
df = pd.read_csv("single_label.csv")
x_train, x_test, y_train, y_test = model_selection.train_test_split(df["fragment"], df["label"],
                                                                        random_state=100, test_size=0.2,
                                                                        stratify=df["label"])


model_ckp = "saved_model_continued_litlat_weights_epoch6"
mod = AutoModelForSequenceClassification.from_pretrained(model_ckp)
tok = AutoTokenizer.from_pretrained(model_ckp)
df_predictions = extract_predictions(x_test, y_test, mod, tok)
df_predictions.to_csv("best_predictions_litlat.csv", index=False)
print("LitLat")
print(df_predictions.head())

### Extract raw embedding for logreg

In [ ]:
# Extract raw embeddings
seed = 100
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

path = "EMBEDDIA/litlat-bert"
tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModel.from_pretrained(path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

df = pd.read_csv("single_label.csv")
x_train, x_test, y_train, y_test = model_selection.train_test_split(df["fragment"], df["label"],
                                                                        random_state=100, test_size=0.2,
                                                                        stratify=df["label"])

x_train, x_val, y_train, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.1,
    random_state=seed)

# input = tokenizer(x_train.to_list(), padding=True, truncation=True, return_tensors="pt")
input_text = x_train.to_list()
batch_size = 128
input_ids = []
attention_mask = []
for i in range(0, len(input_text), batch_size):
    input_batch = input_text[i:i+batch_size]
    input = tokenizer.batch_encode_plus(input_batch, padding="max_length", truncation=True, 
                                        return_tensors="pt", add_special_tokens=True, max_length=512)
    input_ids_batch = input["input_ids"]
    attention_mask_batch = input["attention_mask"]
    input_ids.append(input_ids_batch)
    attention_mask.append(attention_mask_batch)


input_ids = torch.cat(input_ids, dim=0)
attention_mask = torch.cat(attention_mask, dim=0)

# is of dim training set size x embed dim: torch.Size([9743, 512])
print(input_ids.shape)
print(attention_mask.shape)

# Bottleneck, do batching
# embedds are of shape size x 768: torch.Size([100, 768])
embeddings_list = []
for i in range(0, input_ids.shape[0], batch_size):
    with torch.no_grad():
        batch_input_ids = input_ids[i:i+batch_size, :].to(device)
        batch_attention_mask = attention_mask[i:i+batch_size, :].to(device)
        outputs = model(batch_input_ids, batch_attention_mask)
        embeddings = outputs.last_hidden_state[:, 0, :]
        embeddings = embeddings.cpu()
        embeddings_list.append(embeddings)

embeddings_final = torch.cat(embeddings_list, dim=0)

print(embeddings_final.shape)

torch.save({
    "embeddings": embeddings_final,
    "labels": torch.tensor(y_train.values)
}, "train_raw_embeddings.pt")

# implementing batching
seed = 100
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

path = "EMBEDDIA/litlat-bert"
tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModel.from_pretrained(path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


input_text = x_test.to_list()
batch_size = 128
input_ids = []
attention_mask = []
for i in range(0, len(input_text), batch_size):
    input_batch = input_text[i:i+batch_size]
    input = tokenizer.batch_encode_plus(input_batch, padding="max_length", truncation=True, 
                                        return_tensors="pt", add_special_tokens=True, max_length=512)
    input_ids_batch = input["input_ids"]
    attention_mask_batch = input["attention_mask"]
    input_ids.append(input_ids_batch)
    attention_mask.append(attention_mask_batch)


input_ids = torch.cat(input_ids, dim=0)
attention_mask = torch.cat(attention_mask, dim=0)

# is of dim training set size x embed dim: torch.Size([9743, 512])
print(input_ids.shape)
print(attention_mask.shape)

# Bottleneck, do batching
# embedds are of shape size x 768: torch.Size([100, 768])
embeddings_list = []
for i in range(0, input_ids.shape[0], batch_size):
    with torch.no_grad():
        batch_input_ids = input_ids[i:i+batch_size, :].to(device)
        batch_attention_mask = attention_mask[i:i+batch_size, :].to(device)
        outputs = model(batch_input_ids, batch_attention_mask)
        embeddings = outputs.last_hidden_state[:, 0, :]
        embeddings = embeddings.cpu()
        embeddings_list.append(embeddings)

embeddings_final = torch.cat(embeddings_list, dim=0)

print(embeddings_final.shape)

torch.save({
    "embeddings": embeddings_final,
    "labels": torch.tensor(y_test.values)
}, "test_raw_embeddings.pt")

##### Perform balanced logreg

In [ ]:
data = torch.load("train_raw_embeddings.pt")
x_train_emb = data["embeddings"].numpy()
y_train = data["labels"].numpy()

data = torch.load("test_raw_embeddings.pt")
x_test_emb = data["embeddings"].numpy()
y_test = data["labels"].numpy()
print(x_train_emb.shape)
print(y_train.shape)
print(x_test_emb.shape)
print(y_test.shape)

# Baseline
clf = LogisticRegression(max_iter=10000, class_weight="balanced")
clf.fit(x_train_emb, y_train)
predictions = clf.predict(x_test_emb)

accuracy = metrics.accuracy_score(y_pred=predictions, y_true=y_test)
f1 = metrics.f1_score(y_true=y_test, y_pred=predictions, average="macro")
conf_mat = metrics.confusion_matrix(y_test, predictions)

print("Baseline Logistic Regression using raw litlat embeddings")
print("Accuracy", accuracy)
print("F1", f1)
display_cm(conf_mat)

print(classification_report(y_test, predictions, digits=4))